In [1]:
!pip install streamlit  # optional (only needed if you plan to use the original version)
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install cohere
!pip install PyPDF2
!pip install numpy
!pip install pandas
!pip install pypdf

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^



   ------------- -------------------------- 1/3 [fastavro]
   ------------- -------------------------- 1/3 [fastavro]
   ------------- -------------------------- 1/3 [fastavro]
   ------------- -------------------------- 1/3 [fastavro]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------------- ------------- 2/3 [cohere]
   -------------------

In [2]:
import os
import getpass

os.environ["COHERE_API_KEY"] = getpass.getpass("Enter Cohere API Key: ")

In [4]:
import numpy as np
import pandas as pd
import faiss
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from cohere import ClientV2

# Initialize Sentence Transformer model
model_name = "all-MiniLM-L6-v2"  # Adjust model as needed
sentence_transformer_model = SentenceTransformer(model_name)

# Load and process the PDF file
pdf_path = 'Resume.pdf'
loader = PyPDFLoader(pdf_path)
documents = loader.load()

embeddings = []
documents_text = []
sources = []

# To chunkify the docs
latex_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

for docu in documents:  # one page at a time from the PDF
    docs = latex_splitter.create_documents([docu.page_content])  # splitting into chunks
    for document in docs:
        document_embedding = sentence_transformer_model.encode(document.page_content)
        embeddings.append(document_embedding)
        documents_text.append(document.page_content)
        sources.append("www.rheadata.com")

# Create FAISS index
embedding_dimension = len(embeddings[0])
index = faiss.IndexFlatL2(embedding_dimension)
index.add(np.array(embeddings, dtype='float32'))

# Save index and document details
if not os.path.exists("Vector_Store"):
    os.makedirs("Vector_Store")

df = pd.DataFrame({'documents': documents_text, 'source': sources})
df.to_csv('Vector_Store/docs.csv', index=False)
faiss.write_index(index, 'Vector_Store/vector_db.index')

print("✅ FAISS index and document store created successfully!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ FAISS index and document store created successfully!


In [5]:
embeddings

[array([-7.97288343e-02, -2.46599633e-02, -3.26085580e-03, -1.09769115e-02,
        -2.87745502e-02, -1.26787409e-01,  5.62879704e-02,  5.76917566e-02,
        -4.71511558e-02,  2.38866061e-02, -2.86679007e-02, -7.89131671e-02,
         6.88518286e-02, -3.47782448e-02,  9.55017731e-02,  8.28925818e-02,
         6.20789491e-02, -1.24853142e-01,  3.96615900e-02, -5.27570136e-02,
         1.73899401e-02,  4.88848425e-02,  6.25092685e-02, -6.05802834e-02,
         4.00394239e-02,  3.64049934e-02,  1.99846122e-02, -7.39011317e-02,
        -1.86874461e-03, -3.97809073e-02,  2.50412989e-02,  1.30110355e-02,
         3.29486951e-02,  7.17399120e-02, -1.19954022e-02,  4.63455655e-02,
        -2.28512865e-02, -5.14771603e-03,  7.24276155e-02, -4.65678647e-02,
        -1.33263066e-01, -1.23041524e-02, -4.26223502e-03, -2.93950923e-02,
         1.17537624e-03, -7.39392415e-02, -5.36198989e-02,  3.97002026e-02,
         2.16019489e-02, -1.90548543e-02, -1.20993890e-01, -4.03206311e-02,
         5.6

In [8]:

# --- Q&A Section ---

# Load the FAISS index and CSV (optional reload)
index = faiss.read_index('Vector_Store/vector_db.index')
df = pd.read_csv('Vector_Store/docs.csv')

# Initialize Cohere client
co = ClientV2(api_key=os.environ["COHERE_API_KEY"])

# User query input
query = input("Enter your query: ")

if query:
    query_embedding = sentence_transformer_model.encode(query).reshape(1, -1)
    distances, indices = index.search(query_embedding, k=50)

    threshold = 1.7  # Define threshold for distance match
    print("Distance score:", distances)

    if distances[0][0] > threshold:
        print(" Please ask a relevant question.")
    else:
        combined_similar_documents_content_list = []
        similar_documents_sources = []

        for i in indices[0]:
            if i == -1:
                continue
            similar_document_content = df.loc[i, 'documents']
            combined_similar_documents_content_list.append(similar_document_content)

            similar_document_source = df.loc[i, 'source']
            similar_documents_sources.append(similar_document_source)

        combined_similar_documents_content = ' '.join(combined_similar_documents_content_list)

        cohere_prompt = (
            f"Based on the document content: {combined_similar_documents_content}, "
            f"answer the question: '{query}'"
        )

        # Call Cohere API
        cohere_response = co.chat(
            model="command-a-03-2025",
            messages=[{"role": "user", "content": cohere_prompt}],
            temperature=0.3
        )

        print("\n Bot Response:")
        print(cohere_response.message.content[0].text)

        print("\n Sources:")
        print(list(set(similar_documents_sources)))


Distance score: [[1.3617237e+00 1.5507295e+00 1.5926483e+00 1.6106672e+00 1.6205225e+00
  1.6516550e+00 1.6727796e+00 1.6949780e+00 1.8069456e+00 1.8349048e+00
  1.8671139e+00 1.8788618e+00 1.9222312e+00 1.9553680e+00 1.9576442e+00
  3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38
  3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38
  3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38
  3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38
  3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38
  3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38
  3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38 3.4028235e+38]]

 Bot Response:
Based on the document content, the **education** details are as follows:

1. **University of Liverpool, Liverpool, United Kingdom**  
   - **Degree**: Master of Science in Business Analytics and Big Data  
   - **CGPA**: 6.7  
   - **Duration**: 